# 阶段 3：HuggingFace Hub 模型与数据集管理

## 学习目标
- [ ] 掌握模型下载和本地缓存管理
- [ ] 理解 HuggingFace 缓存目录结构
- [ ] 在国内网络环境下正常使用 HF 生态（镜像配置）

## 前置知识
- Python 文件系统操作

## 预计时间
30-60 分钟（不含模型下载时间）

## ⚠️ 重要：先配置镜像
```bash
export HF_ENDPOINT=https://hf-mirror.com
```
或在 notebook 第一个 cell 执行（需在 import 之前）：
```python
import os; os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
```


抱抱脸的模型都存在本地什么位置

In [ ]:
import os

# ====== 第一步：配置镜像（国内用户必须） ======
# HuggingFace 官网在国内访问很慢，这里用 hf-mirror.com 镜像
# 必须在 import huggingface_hub 之前设置，否则无效
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
# 备用镜像（若 hf-mirror.com 也慢）：
# os.environ['HF_ENDPOINT'] = 'https://mirrors.tuna.tsinghua.edu.cn/hugging-face-models'

from huggingface_hub import snapshot_download  # 用于下载整个模型仓库
from tqdm.auto import tqdm                    # 进度条

# ====== 下载模型 ======
# snapshot_download 会下载整个模型仓库（权重、配置、tokenizer 等）
# 并缓存到 ~/.cache/huggingface/hub/ 目录
# 再次运行时会自动跳过已下载的文件
model_path = snapshot_download(
    repo_id='Qwen/Qwen2.5-3B-Instruct',  # HuggingFace Hub 上的仓库 ID
    local_dir_use_symlinks=False,          # Windows 用户必须 False；Mac/Linux 可以用 True（省磁盘）
    # ignore_patterns=['*.bin'],           # 可选：排除旧格式（只保留 safetensors）
)

print(f'模型已缓存到：{model_path}')
print('提示：下次加载这个模型时，可以直接用 model_path 作为 model_name，无需联网')

In [ ]:
from huggingface_hub import scan_cache_dir, HfApi

# ====== 扫描本地缓存 ======
# scan_cache_dir() 读取 ~/.cache/huggingface/ 目录，返回所有已缓存的模型信息
cache_info = scan_cache_dir()  # 返回 HFCacheInfo 对象
api = HfApi()                  # 用于查询模型在 Hub 上的状态（可选）

print(f'=== 本地 HuggingFace 缓存概览 ===')
print(f'缓存目录：{cache_info.cache_dir}')
print(f'总占用空间：{cache_info.size_on_disk / 1e9:.2f} GB')
print(f'已缓存模型数量：{len(list(cache_info.repos))}')
print()

# 遍历所有已缓存的模型/数据集
for repo in cache_info.repos:
    print(f'--- {repo.repo_id} ---')
    print(f'  本地路径：{repo.repo_path}')
    print(f'  占用空间：{repo.size_on_disk / 1e9:.2f} GB')
    print(f'  版本数量：{len(list(repo.revisions))}')
    
    # 查询 Hub 可访问性（需要网络）
    try:
        api.repo_info(repo_id=repo.repo_id)
        print(f'  Hub 状态：可访问')
    except Exception as e:
        print(f'  Hub 状态：不可访问或需要认证')
    print()


In [12]:
import os
from pathlib import Path
from huggingface_hub import scan_cache_dir
from datasets import load_dataset, load_from_disk  # HuggingFace datasets 库
import pandas as pd

# ====== HuggingFace 数据集缓存目录 ======
# 数据集默认缓存在 ~/.cache/huggingface/datasets/
# 模型缓存在 ~/.cache/huggingface/hub/（不同目录）
dataset_cache_dir = os.path.expanduser('~/.cache/huggingface/datasets')
print(f'数据集缓存目录：{dataset_cache_dir}')

if os.path.exists(dataset_cache_dir):
    # 列出所有已缓存的数据集
    datasets = list(Path(dataset_cache_dir).iterdir())
    print(f'已缓存数据集数量：{len(datasets)}')
    for d in datasets[:10]:  # 只显示前 10 个
        size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1e6
        print(f'  - {d.name}（{size:.1f} MB）')
else:
    print('数据集缓存目录不存在，尚未下载任何数据集')

print()

# ====== 加载数据集示例 ======
# 方式 1：从网络加载（首次下载，后续自动使用缓存）
# dataset = load_dataset('openai/gsm8k', 'main')  # GSM8K 数学推理数据集

# 方式 2：从本地缓存加载（离线使用）
# dataset = load_from_disk('~/.cache/huggingface/datasets/gsm8k')

# 方式 3：从 pandas DataFrame 创建（自定义数据）
df = pd.DataFrame({
    'input': ['What is 2+2?', 'What is the capital of France?'],
    'output': ['4', 'Paris']
})
from datasets import Dataset
custom_dataset = Dataset.from_pandas(df)  # 转换为 HuggingFace Dataset 格式
print(f'自定义数据集：{custom_dataset}')
print(f'第一条数据：{custom_dataset[0]}')
print()
print('💡 提示：自定义数据集可以通过 dataset.save_to_disk(path) 保存到本地')


数据集缓存目录: C:\Users\jabel/.cache/huggingface/datasets

本地数据集信息:

数据集: openai/gsm8k
配置: main
路径: C:\Users\jabel\.cache\huggingface\datasets\openai___gsm8k\main\0.0.0\e53f048856ff4f594e959d75785d2c2d37b678ee
大小: 8.71 MB

描述: ...
引用: ...

数据样例(前10条):

--- 数据 1 ---
question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72
----------------------------------------

--- 数据 2 ---
question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
answer: Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10
----------------------------------------

--- 数据 3 ---
question: Betty is saving money for a new wallet which costs $100. Betty has onl

## 本章小结

**关键概念**：
- HuggingFace 缓存在 `~/.cache/huggingface/hub/`，下载一次永久复用
- `scan_cache_dir()` 可以查看所有已缓存的模型和占用空间
- `local_dir_use_symlinks=False` 是 Windows 的必要设置

## 下一步
→ **阶段 4**：[04_qwen25_grpo_finetuning.ipynb](04_qwen25_grpo_finetuning.ipynb)（用下载好的模型做 GRPO 微调）
